# Оценка моделей: влияние загрязнения воздуха на ОПЖ в регионах РФ

Кросс-секция 2015 г., 85 субъектов РФ. Ноутбук оценивает baseline OLS, первую ступень, reduced form, baseline 2SLS, диагностические тесты и несколько robustness-checks.

Ключевое изменение относительно ранней версии: `age_struct_share` не используется в preferred baseline. Эта переменная оставлена только как robustness-control, потому что возрастная структура может частично отражать уже накопленные различия в смертности и миграции. `visits_per_capita` также не включается в baseline: обращаемость в медучреждения может быть эндогенной по отношению к здоровью и измерена шумно.

Все выводы ниже формулируются осторожно: тесты могут не отвергать ограничения, но не доказывают валидность инструментов.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from linearmodels.iv import IV2SLS

pd.set_option('display.float_format', lambda v: f'{v:.4f}')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

## 1. Данные и проверка AO

В папке `final/` может лежать старая версия `validated_dataset_extended.csv`, где Архангельская и Тюменская области записаны как totals “с АО”, а НАО / ХМАО / ЯНАО присутствуют отдельно. Это создаёт двойной счёт. Поэтому ноутбук явно выбирает AO-fixed версию датасета, если она доступна.

In [2]:
def ao_fixed(df):
    vals = df.set_index('region')
    try:
        arch = float(vals.loc['Архангельская область', 'population_avg'])
        tyum = float(vals.loc['Тюменская область', 'population_avg'])
    except KeyError:
        return False
    return np.isclose(arch, 1094549.0) and np.isclose(tyum, 1462199.0)

candidates = [
    Path('../validated_dataset_extended.csv'),  # if notebook is run from final/
    Path('validated_dataset_extended.csv'),     # if notebook is run from project root
    Path('final/validated_dataset_extended.csv')
]
existing = [p for p in candidates if p.exists()]
if not existing:
    raise FileNotFoundError('validated_dataset_extended.csv не найден ни в project root, ни в final/.')

chosen = None
for p in existing:
    tmp = pd.read_csv(p)
    if ao_fixed(tmp):
        chosen = p
        df = tmp
        break

if chosen is None:
    chosen = existing[0]
    df = pd.read_csv(chosen)
    print('ВНИМАНИЕ: AO-fixed версия датасета не найдена; проверьте preprocessing перед финальной сдачей.')

print(f'Загружен датасет: {chosen.resolve()}')
print('shape:', df.shape)

check_regions = [
    'Архангельская область',
    'Ненецкий автономный округ (Архангельская область)',
    'Тюменская область',
    'Ханты-Мансийский автономный округ - Югра (Тюменская область)',
    'Ямало-Ненецкий автономный округ (Тюменская область)',
]
df.loc[df['region'].isin(check_regions),
       ['region', 'population_avg', 'grp_total', 'life_expectancy', 'age_struct_share']].sort_values('region')

Загружен датасет: /Users/maria/Desktop/Code/HSE/METRICS-2/project/validated_dataset_extended.csv
shape: (85, 34)


,region,population_avg,grp_total,life_expectancy,age_struct_share
2,Архангельская область,1094549.0000,400504517.7000,70.3300,50.9000
32,Ненецкий автономный округ (Архангельская область),42389.0000,227193534.4000,70.2900,55.0000
74,Тюменская область,1462199.0000,905673481.7000,70.9100,54.9000
78,Ханты-Мансийский автономный округ - Югра (Тюме...,1624107.0000,3154058740.2000,72.6200,61.6000
83,Ямало-Ненецкий автономный округ (Тюменская обл...,519245.0000,1791825606.0000,71.6100,64.7000


## 2. Переменные и спецификации

Preferred baseline использует `ln_grp_pc`, `urban_share`, `ln_population_avg` как контроли. `age_struct_share` проверяется отдельно в robustness specification. `region` используется только как идентификатор наблюдения, а не как категориальный регрессор.

In [3]:
DV = 'life_expectancy'
ENDO = 'ln_emissions_h1_pc'
IV_MAIN = 'ln_emp_mining_pc'
IV_ALT_ENERGY = 'ln_emp_energy_pc'
IV_ALT_FUEL = 'ln_emissions_fuel_pc'

BASE_CONTROLS = ['ln_grp_pc', 'urban_share', 'ln_population_avg']
AGE_CONTROL = 'age_struct_share'
ROBUST_CONTROLS = BASE_CONTROLS + [AGE_CONTROL]
ALT_IVS = [IV_ALT_ENERGY, IV_ALT_FUEL]

BASE_VARS = [DV, ENDO, IV_MAIN] + ALT_IVS + BASE_CONTROLS
AGE_VARS = [DV, ENDO, IV_MAIN] + ALT_IVS + ROBUST_CONTROLS

data_base = df[['region'] + BASE_VARS].dropna().reset_index(drop=True)
data_age = df[['region'] + AGE_VARS].dropna().reset_index(drop=True)

print(f'Baseline sample: N = {len(data_base)}')
print(f'Age robustness sample: N = {len(data_age)}')
print('Baseline controls:', BASE_CONTROLS)
print('Age robustness controls:', ROBUST_CONTROLS)

df[[DV, ENDO, IV_MAIN] + BASE_CONTROLS + [AGE_CONTROL]].describe().T.round(4)

Baseline sample: N = 85
Age robustness sample: N = 85
Baseline controls: ['ln_grp_pc', 'urban_share', 'ln_population_avg']
Age robustness controls: ['ln_grp_pc', 'urban_share', 'ln_population_avg', 'age_struct_share']


,count,mean,std,min,25%,50%,75%,max
life_expectancy,85.0000,70.5544,2.3935,63.1400,69.1400,70.3900,71.6300,78.4700
ln_emissions_h1_pc,85.0000,-10.3361,1.3930,-14.5157,-11.2033,-10.2621,-9.4985,-6.9957
ln_emp_mining_pc,85.0000,-5.5707,1.6549,-8.7341,-6.7567,-5.9033,-4.5023,-1.5260
ln_grp_pc,85.0000,5.8338,0.6718,4.6834,5.4083,5.7689,6.0190,8.5867
urban_share,85.0000,70.0859,13.2474,29.9000,63.7000,71.3000,77.6000,100.0000
ln_population_avg,85.0000,13.9603,0.9707,10.6546,13.5131,13.9885,14.6632,16.3233
age_struct_share,85.0000,58.0471,2.1448,50.9000,56.8000,57.7000,59.1000,64.9000


## 3. Baseline OLS

OLS не трактуется как причинная оценка: она нужна как отправная точка для знака и масштаба связи между загрязнением и ОПЖ.

In [4]:
X_ols = sm.add_constant(data_base[[ENDO] + BASE_CONTROLS])
ols = sm.OLS(data_base[DV], X_ols).fit(cov_type='HC1')
print(ols.summary())

b_ols = ols.params[ENDO]
se_ols = ols.bse[ENDO]
print(f'\nOLS beta({ENDO}) = {b_ols:+.4f}, HC1 SE = {se_ols:.4f}, p = {ols.pvalues[ENDO]:.4g}')
print(f'Удвоение выбросов связано с изменением ОПЖ на {b_ols * np.log(2):+.3f} года.')

                            OLS Regression Results                            
Dep. Variable:        life_expectancy   R-squared:                       0.517
Model:                            OLS   Adj. R-squared:                  0.493
Method:                 Least Squares   F-statistic:                     19.11
Date:                Sun, 24 May 2026   Prob (F-statistic):           4.60e-11
Time:                        13:02:35   Log-Likelihood:                -163.37
No. Observations:                  85   AIC:                             336.7
Df Residuals:                      80   BIC:                             348.9
Df Model:                           4                                         
Covariance Type:                  HC1                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 36.4870      4

## 4. Первая ступень 2SLS

Проверяется relevance preferred instrument `ln_emp_mining_pc`. Порог `F > 10` используется только как правило большого пальца; для robust F-статистик сравнение со Stock-Yogo нужно трактовать эвристически.

In [5]:
X_fs = sm.add_constant(data_base[[IV_MAIN] + BASE_CONTROLS])
fs = sm.OLS(data_base[ENDO], X_fs).fit(cov_type='HC1')
F_test = fs.f_test(f'{IV_MAIN} = 0')
F_stat = float(np.squeeze(F_test.fvalue))
F_pval = float(F_test.pvalue)
F_df1, F_df2 = int(F_test.df_num), int(F_test.df_denom)

print(fs.summary())
print('\nFirst-stage relevance test')
print(f'H0: coefficient on {IV_MAIN} = 0')
print(f'F({F_df1}, {F_df2}) = {F_stat:.4f}, p = {F_pval:.4g}')
print('Interpretation: инструмент статистически релевантен, но сила пограничная / умеренная, не явно strong.')

                            OLS Regression Results                            
Dep. Variable:     ln_emissions_h1_pc   R-squared:                       0.516
Model:                            OLS   Adj. R-squared:                  0.491
Method:                 Least Squares   F-statistic:                     30.25
Date:                Sun, 24 May 2026   Prob (F-statistic):           2.47e-15
Time:                        13:02:35   Log-Likelihood:                -117.47
No. Observations:                  85   AIC:                             244.9
Df Residuals:                      80   BIC:                             257.2
Df Model:                           4                                         
Covariance Type:                  HC1                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const               -13.0628      3.52

## 5. Reduced Form

Reduced form показывает связь инструмента с зависимой переменной после тех же baseline controls. Ожидаемый знак отрицательный: больше добычи -> больше загрязнения -> ниже ОПЖ. Это не доказывает exclusion restriction, но полезно для общей IV-логики.

In [6]:
X_rf = sm.add_constant(data_base[[IV_MAIN] + BASE_CONTROLS])
rf = sm.OLS(data_base[DV], X_rf).fit(cov_type='HC1')
print(rf.summary())
print(f'\nReduced-form coefficient on {IV_MAIN}: {rf.params[IV_MAIN]:+.4f}, p = {rf.pvalues[IV_MAIN]:.4g}')

                            OLS Regression Results                            
Dep. Variable:        life_expectancy   R-squared:                       0.305
Model:                            OLS   Adj. R-squared:                  0.270
Method:                 Least Squares   F-statistic:                     7.015
Date:                Sun, 24 May 2026   Prob (F-statistic):           6.81e-05
Time:                        13:02:35   Log-Likelihood:                -178.85
No. Observations:                  85   AIC:                             367.7
Df Residuals:                      80   BIC:                             379.9
Df Model:                           4                                         
Covariance Type:                  HC1                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                51.1695      7.14

## 6. Baseline 2SLS

Baseline IV использует `ln_emp_mining_pc` как preferred instrument. Причинная интерпретация остаётся осторожной из-за пограничной силы инструмента и cross-section дизайна.

In [7]:
exog_base = sm.add_constant(data_base[BASE_CONTROLS])
iv = IV2SLS(
    dependent=data_base[DV],
    exog=exog_base,
    endog=data_base[[ENDO]],
    instruments=data_base[[IV_MAIN]],
).fit(cov_type='robust', debiased=True)

iv_table = pd.DataFrame({
    'coef': iv.params,
    'robust_se': iv.std_errors,
    't': iv.tstats,
    'p': iv.pvalues,
})
print(iv_table.round(4).to_string())
print(f'R^2 = {iv.rsquared:.4f}')

b_iv = iv.params[ENDO]
print(f'\n2SLS beta({ENDO}) = {b_iv:+.4f}, robust SE = {iv.std_errors[ENDO]:.4f}, p = {iv.pvalues[ENDO]:.4g}')
print(f'Удвоение выбросов связано с изменением ОПЖ на {b_iv * np.log(2):+.3f} года.')

                      coef  robust_se       t      p
const              30.4784     9.7992  3.1103 0.0026
ln_grp_pc           2.2909     0.8110  2.8247 0.0060
urban_share        -0.0451     0.0153 -2.9511 0.0042
ln_population_avg   0.9672     0.2323  4.1642 0.0001
ln_emissions_h1_pc -1.5840     0.4802 -3.2983 0.0015
R^2 = 0.4900

2SLS beta(ln_emissions_h1_pc) = -1.5840, robust SE = 0.4802, p = 0.001453
Удвоение выбросов связано с изменением ОПЖ на -1.098 года.


## 7. Сводная Таблица: OLS vs 2SLS

Таблица ниже годится как основа для вставки в текст проекта через экспорт в Word/TeX, а не как скриншот `summary()`.

In [8]:
def fmt_coef(coef, se, p):
    stars = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
    return f'{coef:+.4f}{stars} ({se:.4f})'

rows = []
for v in ['const', ENDO] + BASE_CONTROLS:
    rows.append({
        'variable': v,
        'OLS baseline': fmt_coef(ols.params[v], ols.bse[v], ols.pvalues[v]),
        '2SLS baseline': fmt_coef(iv.params[v], iv.std_errors[v], iv.pvalues[v]),
    })
rows += [
    {'variable': 'N', 'OLS baseline': int(ols.nobs), '2SLS baseline': int(iv.nobs)},
    {'variable': 'R^2', 'OLS baseline': f'{ols.rsquared:.4f}', '2SLS baseline': f'{iv.rsquared:.4f}'},
    {'variable': 'Robust SE', 'OLS baseline': 'HC1', '2SLS baseline': 'robust, debiased'},
]
main_table = pd.DataFrame(rows)
main_table

,variable,OLS baseline,2SLS baseline
0,const,+36.4870*** (4.9334),+30.4784*** (9.7992)
1,ln_emissions_h1_pc,-1.2065*** (0.2082),-1.5840*** (0.4802)
2,ln_grp_pc,+1.8119*** (0.5004),+2.2909*** (0.8110)
3,urban_share,-0.0474*** (0.0152),-0.0451*** (0.0153)
4,ln_population_avg,+1.0276*** (0.2444),+0.9672*** (0.2323)
5,N,85,85
6,R^2,0.5169,0.4900
7,Robust SE,HC1,"robust, debiased"


## 8. Диагностика: Wu-Hausman / Durbin

`H0`: `ln_emissions_h1_pc` экзогенна. Если `H0` не отвергается, это не доказывает экзогенность, а означает, что тест не даёт достаточно evidence против OLS.

In [9]:
wh = iv.wu_hausman()
du = iv.durbin()

diag_exog = pd.DataFrame([
    {'test': 'Wu-Hausman', 'H0': f'{ENDO} экзогенна', 'stat': wh.stat, 'distribution': wh.dist_name, 'p_value': wh.pval},
    {'test': 'Durbin', 'H0': f'{ENDO} экзогенна', 'stat': du.stat, 'distribution': du.dist_name, 'p_value': du.pval},
])
diag_exog.round(4)

,test,H0,stat,distribution,p_value
0,Wu-Hausman,ln_emissions_h1_pc экзогенна,0.6806,"F(1,79)",0.4119
1,Durbin,ln_emissions_h1_pc экзогенна,0.7260,chi2(1),0.3942


**Интерпретация.** С preferred mining-IV тесты обычно дают слабое evidence of endogeneity. Это не делает OLS автоматически “правильной”: тест может иметь низкую мощность при N = 85 и пограничной силе инструмента. Поэтому вывод: evidence on endogeneity is mixed, а causal interpretation остаётся осторожной.

## 9. Сила Инструмента

`linearmodels` и `statsmodels` используют близкие, но не полностью одинаковые реализации robust covariance и распределения статистики, поэтому first-stage partial F может немного отличаться от `sm.OLS.f_test`.

In [10]:
fs_diag = iv.first_stage.diagnostics.loc[ENDO]
print(iv.first_stage)
print('\nСравнение first-stage статистик:')
print(f'sm.OLS HC1 F-test: F = {F_stat:.4f}, p = {F_pval:.4g}')
print(f'linearmodels partial F/Wald: {float(fs_diag["f.stat"]):.4f}, dist = {fs_diag["f.dist"]}, p = {float(fs_diag["f.pval"]):.4g}')

        First Stage Estimation Results       
                           ln_emissions_h1_pc
---------------------------------------------
R-squared                              0.5156
Partial R-squared                      0.1328
Shea's R-squared                       0.1328
Partial F-statistic                    8.5414
P-value (Partial F-stat)               0.0035
Partial F-stat Distn                  chi2(1)
==========================        ===========
const                                 -13.063
                                    (-3.7103)
ln_grp_pc                              0.7797
                                     (2.4065)
urban_share                            0.0111
                                     (0.9272)
ln_population_avg                     -0.0649
                                    (-0.4655)
ln_emp_mining_pc                       0.3035
                                     (2.9226)
---------------------------------------------

T-stats reported in parentheses
T

**Вывод по first stage.** Инструмент статистически релевантен, но его сила не выглядит бесспорно высокой. Пороги Stock-Yogo полезны как ориентир, но для robust F-статистик их нужно использовать осторожно. Для уровня undergraduate project достаточно зафиксировать: instrument appears moderately / borderline strong, not clearly strong.

## 10. VIF

Проверяем мультиколлинеарность в preferred baseline. `region` не включается как dummy-переменная.

In [11]:
vif_vars = [ENDO] + BASE_CONTROLS
X_vif = sm.add_constant(data_base[vif_vars])
vif_tab = pd.DataFrame({
    'variable': ['const'] + vif_vars,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
vif_tab.round(3)

,variable,VIF
0,const,585.2360
1,ln_emissions_h1_pc,1.7910
2,ln_grp_pc,2.1840
3,urban_share,1.5680
4,ln_population_avg,1.1750


## 11. Overidentification: Sargan

Тест Саргана показывается только для сверхидентифицированных спецификаций. Нулевая гипотеза “overidentifying restrictions hold”. Неотвержение H0 не доказывает валидность инструментов, особенно если инструменты могут нарушать exclusion restriction похожим образом.

In [12]:
iv_over = IV2SLS(
    dependent=data_base[DV],
    exog=exog_base,
    endog=data_base[[ENDO]],
    instruments=data_base[[IV_MAIN, IV_ALT_ENERGY]],
).fit(cov_type='robust', debiased=True)
sg = iv_over.sargan

pd.DataFrame([{
    'spec': 'mining + energy',
    'stat': sg.stat,
    'df': sg.df,
    'p_value': sg.pval,
    'interpretation': 'restrictions not rejected; not proof of IV validity',
}]).round(4)

,spec,stat,df,p_value,interpretation
0,mining + energy,0.8726,1,0.3502,restrictions not rejected; not proof of IV val...


## 12. Влиятельные Наблюдения

Baseline оставляет все наблюдения. Diagnostics только помечают регионы с высоким leverage / Cook's D. Удаление используется отдельно как robustness-check, а не как preferred specification.

In [13]:
# Для диагностики влияния используем спецификацию с age_struct_share: именно в ней
# Ненецкий АО наиболее явно проявляется как influential observation.
X_age_ols = sm.add_constant(data_age[[ENDO] + ROBUST_CONTROLS])
ols_age_plain = sm.OLS(data_age[DV], X_age_ols).fit()
infl = ols_age_plain.get_influence()
cooks, _ = infl.cooks_distance
lev = infl.hat_matrix_diag
n_obs = int(ols_age_plain.nobs)
k = X_age_ols.shape[1]
cook_thr = 4.0 / n_obs
lev_thr = 2.0 * k / n_obs

diag = pd.DataFrame({
    'region': data_age['region'],
    'cooks_D': cooks,
    'leverage': lev,
})
diag['flag_cook_4n'] = diag['cooks_D'] > cook_thr
diag['flag_leverage'] = diag['leverage'] > lev_thr
diag['flag_severe_cook'] = diag['cooks_D'] > 1.0
flagged = diag[(diag['flag_cook_4n']) | (diag['flag_leverage'])].sort_values('cooks_D', ascending=False)

print(f'Пороги: Cook D > 4/n = {cook_thr:.4f}; leverage > 2k/n = {lev_thr:.4f}')
print(f'Помечено наблюдений: {len(flagged)} из {n_obs}')
flagged.head(15).round(4)

Пороги: Cook D > 4/n = 0.0471; leverage > 2k/n = 0.1412
Помечено наблюдений: 15 из 85


,region,cooks_D,leverage,flag_cook_4n,flag_leverage,flag_severe_cook
32,Ненецкий автономный округ (Архангельская область),0.9614,0.4960,True,True,False
82,Чукотский автономный округ,0.4703,0.2465,True,True,False
48,Республика Ингушетия,0.2734,0.1597,True,True,False
58,Республика Тыва,0.1248,0.0691,True,False,False
65,Сахалинская область,0.0927,0.1204,True,False,False
18,Карачаево-Черкесская Республика,0.0814,0.0876,True,False,False
78,Ханты-Мансийский автономный округ - Югра (Тюме...,0.0493,0.1046,True,False,False
2,Архангельская область,0.0490,0.1811,True,True,False
44,Республика Алтай,0.0474,0.1496,True,True,False
83,Ямало-Ненецкий автономный округ (Тюменская обл...,0.0427,0.2026,False,True,False


In [14]:
nao_name = 'Ненецкий автономный округ (Архангельская область)'
nao_diag = diag.loc[diag['region'] == nao_name]
print(nao_diag.round(4).to_string(index=False))

                                           region  cooks_D  leverage  flag_cook_4n  flag_leverage  flag_severe_cook
Ненецкий автономный округ (Архангельская область)   0.9614    0.4960          True           True             False


**Интерпретация.** Ненецкий АО остаётся наиболее влиятельным наблюдением: в старой AO-inclusive сборке его Cook's D мог превышать 1, а после AO-fixed пересчёта точное значение выводится выше и обычно остаётся близким к порогу 1. Высокий leverage связан с малым населением и экстремальными per-capita показателями добычи / выбросов. Baseline keeps all observations; это важные регионы для содержательной вариации, а не технический мусор.

## 13. Robustness: `age_struct_share` и Влиятельные Наблюдения

`age_struct_share` добавляется только как robustness-control. Если нет наблюдений с `Cook's D > 1`, дополнительно показывается leave-one-out без Ненецкого АО как sensitivity-check, потому что он имеет самый высокий Cook's D.

In [15]:
def fit_ols_iv(dataset, controls, label):
    x = sm.add_constant(dataset[[ENDO] + controls])
    m_ols = sm.OLS(dataset[DV], x).fit(cov_type='HC1')
    m_iv = IV2SLS(
        dependent=dataset[DV],
        exog=sm.add_constant(dataset[controls]),
        endog=dataset[[ENDO]],
        instruments=dataset[[IV_MAIN]],
    ).fit(cov_type='robust', debiased=True)
    return {
        'spec': label,
        'N': len(dataset),
        'OLS_beta': m_ols.params[ENDO],
        'OLS_p': m_ols.pvalues[ENDO],
        'IV_beta': m_iv.params[ENDO],
        'IV_p': m_iv.pvalues[ENDO],
        'Wu_p': m_iv.wu_hausman().pval,
    }

severe_regions = diag.loc[diag['flag_severe_cook'], 'region'].tolist()
if severe_regions:
    drop_regions = severe_regions
    drop_label = 'drop Cook D > 1'
else:
    drop_regions = [nao_name]
    drop_label = 'leave-one-out: Nenets AO'

trim_age = data_age.loc[~data_age['region'].isin(drop_regions)].reset_index(drop=True)
robust_age = pd.DataFrame([
    fit_ols_iv(data_base, BASE_CONTROLS, 'preferred baseline'),
    fit_ols_iv(data_age, ROBUST_CONTROLS, 'add age_struct_share'),
    fit_ols_iv(trim_age, ROBUST_CONTROLS, drop_label),
])
robust_age.round(4)

,spec,N,OLS_beta,OLS_p,IV_beta,IV_p,Wu_p
0,preferred baseline,85,-1.2065,0.0000,-1.5840,0.0015,0.4119
1,add age_struct_share,85,-1.1735,0.0000,-1.6058,0.0008,0.3240
2,leave-one-out: Nenets AO,84,-1.0861,0.0000,-1.4711,0.0026,0.3982


## 14. Robustness: Альтернативные Инструменты

`ln_emp_energy_pc` и `ln_emissions_fuel_pc` проверяются как альтернативные IV, но не становятся preferred instruments. Они могут быть статистически более релевантными, однако экономически слабее по exclusion restriction: fuel emissions механически связаны с загрязнением, а energy employment может влиять на ОПЖ через доходы и инфраструктуру.

In [16]:
def fit_iv_spec(instr_list, label):
    iv_m = IV2SLS(
        dependent=data_base[DV],
        exog=exog_base,
        endog=data_base[[ENDO]],
        instruments=data_base[instr_list],
    ).fit(cov_type='robust', debiased=True)
    fs_m = sm.OLS(
        data_base[ENDO],
        sm.add_constant(data_base[instr_list + BASE_CONTROLS]),
    ).fit(cov_type='HC1')
    f_expr = ', '.join(f'{x} = 0' for x in instr_list)
    f_val = float(np.squeeze(fs_m.f_test(f_expr).fvalue))
    return {
        'spec': label,
        'instruments': ', '.join(instr_list),
        'beta': iv_m.params[ENDO],
        'SE': iv_m.std_errors[ENDO],
        'p': iv_m.pvalues[ENDO],
        'first_stage_F': f_val,
        'Wu_p': iv_m.wu_hausman().pval,
        'Sargan_p': iv_m.sargan.pval if len(instr_list) > 1 else np.nan,
    }

iv_specs = pd.DataFrame([
    fit_iv_spec([IV_MAIN], 'preferred: mining'),
    fit_iv_spec([IV_ALT_ENERGY], 'alternative: energy'),
    fit_iv_spec([IV_ALT_FUEL], 'alternative: fuel emissions'),
    fit_iv_spec([IV_MAIN, IV_ALT_ENERGY], 'overid: mining + energy'),
])
iv_specs.round(4)

,spec,instruments,beta,SE,p,first_stage_F,Wu_p,Sargan_p
0,preferred: mining,ln_emp_mining_pc,-1.5840,0.4802,0.0015,8.5414,0.4119,NaN
1,alternative: energy,ln_emp_energy_pc,-2.0811,0.5312,0.0002,8.5632,0.0091,NaN
2,alternative: fuel emissions,ln_emissions_fuel_pc,-2.0553,0.3087,0.0000,69.2192,0.0000,NaN
3,overid: mining + energy,"ln_emp_mining_pc, ln_emp_energy_pc",-1.9251,0.4267,0.0000,5.2412,0.0128,0.3502


**Интерпретация альтернативных IV.** Более сильная first stage у alternative IV не делает их предпочтительными. `ln_emissions_fuel_pc` особенно проблемен как инструмент, потому что он является частью pollution channel. Поэтому эти спецификации поддерживают отрицательный знак как robustness evidence, но не “валидируют” causal estimate.

## 15. Гипотезы и Итоговые Тесты

Гипотезы содержательно направленные, но стандартные p-values ниже двусторонние.

In [17]:
def verdict(pval, alpha=0.05):
    return 'H0 отвергается' if pval < alpha else 'H0 не отвергается'

hypotheses = pd.DataFrame([
    {
        'Гипотеза': 'H1: рост выбросов связан со снижением ОПЖ',
        'Проверка': f'OLS beta = {ols.params[ENDO]:+.4f}; 2SLS beta = {iv.params[ENDO]:+.4f}',
        'p-value': f'OLS p = {ols.pvalues[ENDO]:.4g}; 2SLS p = {iv.pvalues[ENDO]:.4g}',
        'Вывод': 'поддерживается по знаку и значимости; causal interpretation cautious',
    },
    {
        'Гипотеза': 'H2: доход положительно связан с ОПЖ',
        'Проверка': f'OLS beta_grp = {ols.params["ln_grp_pc"]:+.4f}; 2SLS beta_grp = {iv.params["ln_grp_pc"]:+.4f}',
        'p-value': f'OLS p = {ols.pvalues["ln_grp_pc"]:.4g}; 2SLS p = {iv.pvalues["ln_grp_pc"]:.4g}',
        'Вывод': 'поддерживается в baseline; age_struct_share только robustness',
    },
    {
        'Гипотеза': 'H3: добыча связана с выбросами',
        'Проверка': f'pi1 = {fs.params[IV_MAIN]:+.4f}, F = {F_stat:.2f}',
        'p-value': f'p = {fs.pvalues[IV_MAIN]:.4g}',
        'Вывод': 'relevance поддерживается; сила инструмента пограничная / умеренная',
    },
])
hypotheses

,Гипотеза,Проверка,p-value,Вывод
0,H1: рост выбросов связан со снижением ОПЖ,OLS beta = -1.2065; 2SLS beta = -1.5840,OLS p = 6.777e-09; 2SLS p = 0.001453,поддерживается по знаку и значимости; causal i...
1,H2: доход положительно связан с ОПЖ,OLS beta_grp = +1.8119; 2SLS beta_grp = +2.2909,OLS p = 0.0002932; 2SLS p = 0.005973,поддерживается в baseline; age_struct_share то...
2,H3: добыча связана с выбросами,"pi1 = +0.3035, F = 8.54",p = 0.003472,relevance поддерживается; сила инструмента пог...


In [18]:
tests = pd.DataFrame([
    {
        'Тест': 'First stage relevance',
        'H0': f'{IV_MAIN} coefficient = 0',
        'Статистика': f'F = {F_stat:.4f}',
        'Распределение': f'F({F_df1}, {F_df2})',
        'p-value': f'{F_pval:.4g}',
        'Решение': verdict(F_pval),
        'Содержательный вывод': 'инструмент релевантен, но не clearly strong',
    },
    {
        'Тест': 'Wu-Hausman',
        'H0': f'{ENDO} экзогенна',
        'Статистика': f'F = {wh.stat:.4f}',
        'Распределение': wh.dist_name,
        'p-value': f'{wh.pval:.4f}',
        'Решение': verdict(wh.pval),
        'Содержательный вывод': 'evidence on endogeneity mixed',
    },
    {
        'Тест': 'Durbin',
        'H0': f'{ENDO} экзогенна',
        'Статистика': f'chi2 = {du.stat:.4f}',
        'Распределение': du.dist_name,
        'p-value': f'{du.pval:.4f}',
        'Решение': verdict(du.pval),
        'Содержательный вывод': 'согласуется с Wu-Hausman',
    },
    {
        'Тест': 'Sargan overidentification',
        'H0': 'overidentifying restrictions hold',
        'Статистика': f'chi2 = {sg.stat:.4f}',
        'Распределение': f'chi2({sg.df})',
        'p-value': f'{sg.pval:.4f}',
        'Решение': verdict(sg.pval),
        'Содержательный вывод': 'restrictions not rejected; validity not proven',
    },
])
tests

,Тест,H0,Статистика,Распределение,p-value,Решение,Содержательный вывод
0,First stage relevance,ln_emp_mining_pc coefficient = 0,F = 8.5414,"F(1, 80)",0.004512,H0 отвергается,"инструмент релевантен, но не clearly strong"
1,Wu-Hausman,ln_emissions_h1_pc экзогенна,F = 0.6806,"F(1,79)",0.4119,H0 не отвергается,evidence on endogeneity mixed
2,Durbin,ln_emissions_h1_pc экзогенна,chi2 = 0.7260,chi2(1),0.3942,H0 не отвергается,согласуется с Wu-Hausman
3,Sargan overidentification,overidentifying restrictions hold,chi2 = 0.8726,chi2(1),0.3502,H0 не отвергается,restrictions not rejected; validity not proven


## 16. Краткий Вывод

- H1 поддерживается: связь `ln_emissions_h1_pc` с ОПЖ отрицательная в OLS и 2SLS.
- H2 поддерживается в baseline: `ln_grp_pc` положителен после контроля загрязнения, урбанизации и размера региона. Формулировку “после контроля на возрастную структуру” лучше оставить только для robustness.
- H3 поддерживается как relevance condition: `ln_emp_mining_pc` положительно связан с выбросами, но first-stage сила пограничная / умеренная.
- Preferred IV остаётся `ln_emp_mining_pc`. Alternative IV useful for robustness, but weaker economically under exclusion restriction.
- Main causal language should remain cautious: results support the hypothesized negative relationship, but do not fully validate a strong causal interpretation.